In [1]:
from google.colab import drive
# Attempt to unmount first to ensure a clean state
drive.flush_and_unmount()

# Remove any lingering files in the mountpoint directory
!rm -rf /content/drive/*

drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [2]:
import geopandas as gpd
from PIL import Image
import os

geojson_path = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_labels/xView_train.geojson"
image_dir = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_images"
output_labels_dir = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/yolo_labels"

os.makedirs(output_labels_dir, exist_ok=True)
gdf = gpd.read_file(geojson_path)

# Remapper les type_id (qui ont des trous) vers des indices YOLO 0..N-1
unique_classes = sorted(gdf['type_id'].unique())
class_to_idx = {c: i for i, c in enumerate(unique_classes)}

for img_id in gdf['image_id'].unique():
    img_path = os.path.join(image_dir, img_id)
    if not os.path.exists(img_path):
        continue
    with Image.open(img_path) as im:
        img_w, img_h = im.size

    rows = gdf[gdf['image_id'] == img_id]
    lines = []
    for _, row in rows.iterrows():
        xmin, ymin, xmax, ymax = map(float, row['bounds_imcoords'].split(','))
        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        cls_idx = class_to_idx[row['type_id']]
        lines.append(f"{cls_idx} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

    txt_name = img_id.replace('.tif', '.txt')
    with open(os.path.join(output_labels_dir, txt_name), 'w') as f:
        f.write('\n'.join(lines))

print(f"{len(unique_classes)} classes, conversion terminée dans {output_labels_dir}")

62 classes, conversion terminée dans /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/yolo_labels


In [3]:
import requests

url = "https://raw.githubusercontent.com/DIUx-xView/data_utilities/master/xview_class_labels.txt"
response = requests.get(url)
official_classes = {}
for line in response.text.strip().split('\n'):
    idx, name = line.split(':')
    official_classes[int(idx)] = name.strip()

extra_ids = set(unique_classes) - set(official_classes.keys())
print(f"IDs présents dans tes données mais absents de la liste officielle : {extra_ids}")

IDs présents dans tes données mais absents de la liste officielle : {np.int32(82), np.int32(75)}


In [4]:
import requests

url = "https://raw.githubusercontent.com/DIUx-xView/data_utilities/master/xview_class_labels.txt"
response = requests.get(url)

class_names = {}
for line in response.text.strip().split('\n'):
    idx, name = line.split(':')
    class_names[int(idx)] = name.strip()

print(len(class_names))

60


In [5]:
extra_ids = set(unique_classes) - set(class_names.keys())
print(f"IDs présents dans tes données mais absents de la liste officielle : {extra_ids}")

for eid in extra_ids:
    count = (gdf['type_id'] == eid).sum()
    print(f"type_id {eid} : {count} occurrences")

IDs présents dans tes données mais absents de la liste officielle : {np.int32(82), np.int32(75)}
type_id 82 : 28 occurrences
type_id 75 : 51 occurrences


In [6]:
gdf_clean = gdf[~gdf['type_id'].isin([75, 82])].copy()
print(f"Annotations avant filtrage : {len(gdf)}")
print(f"Annotations après filtrage : {len(gdf_clean)}")

Annotations avant filtrage : 601937
Annotations après filtrage : 601858


In [7]:
from PIL import Image
import os

image_dir = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_images"
output_labels_dir = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/yolo_labels"

os.makedirs(output_labels_dir, exist_ok=True)

# Utilise le mapping officiel (60 classes), trié par ID pour un index stable
official_ids = sorted(class_names.keys())
class_to_idx = {c: i for i, c in enumerate(official_ids)}

for img_id in gdf_clean['image_id'].unique():
    img_path = os.path.join(image_dir, img_id)
    if not os.path.exists(img_path):
        continue
    with Image.open(img_path) as im:
        img_w, img_h = im.size

    rows = gdf_clean[gdf_clean['image_id'] == img_id]
    lines = []
    for _, row in rows.iterrows():
        xmin, ymin, xmax, ymax = map(float, row['bounds_imcoords'].split(','))
        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        cls_idx = class_to_idx[row['type_id']]
        lines.append(f"{cls_idx} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

    txt_name = img_id.replace('.tif', '.txt')
    with open(os.path.join(output_labels_dir, txt_name), 'w') as f:
        f.write('\n'.join(lines))

print(f"{len(official_ids)} classes officielles, conversion terminée dans {output_labels_dir}")

60 classes officielles, conversion terminée dans /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/yolo_labels


In [8]:
all_images = set(f for f in os.listdir(image_dir) if f.endswith('.tif'))
annotated_images = set(gdf_clean['image_id'].unique())
missing = all_images - annotated_images
print(f"Images sans aucune annotation : {missing}")

Images sans aucune annotation : {'._100.tif', '._102.tif', '._109.tif'}


In [9]:
for img_id in missing:
    txt_name = img_id.replace('.tif', '.txt')
    open(os.path.join(output_labels_dir, txt_name), 'w').close()

print(f"{len(missing)} fichiers vides créés pour les images sans annotation.")

3 fichiers vides créés pour les images sans annotation.


In [10]:
import os

# Supprimer les fichiers parasites macOS
junk_files = ['._100.tif', '._102.tif', '._109.tif']
for f in junk_files:
    path = os.path.join(image_dir, f)
    if os.path.exists(path):
        os.remove(path)
        print(f"Supprimé : {path}")

# Supprimer les .txt vides créés à tort pour ces faux fichiers
for f in junk_files:
    txt_name = f.replace('.tif', '.txt')
    txt_path = os.path.join(output_labels_dir, txt_name)
    if os.path.exists(txt_path):
        os.remove(txt_path)
        print(f"Supprimé : {txt_path}")

Supprimé : /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_images/._100.tif
Supprimé : /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_images/._102.tif
Supprimé : /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/train_images/._109.tif
Supprimé : /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/yolo_labels/._100.txt
Supprimé : /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/yolo_labels/._102.txt
Supprimé : /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/yolo_labels/._109.txt


In [11]:
real_images = [f for f in os.listdir(image_dir) if f.endswith('.tif') and not f.startswith('._')]
real_labels = [f for f in os.listdir(output_labels_dir) if f.endswith('.txt')]
print(f"Images réelles : {len(real_images)}")
print(f"Labels : {len(real_labels)}")

Images réelles : 846
Labels : 846
